# Experiment 2.13: Activation Curvature Theory — Does Nonlinearity Shape Muon's Edge?

## Core Question

Muon replaces raw gradients with their **Newton-Schulz orthogonalized** counterparts, effectively projecting each gradient step onto the nearest orthogonal matrix. This is powerful — but *why* does it help, and when does a simpler orthogonality penalty (`||W^T W - I||_F^2`) fail to replicate that benefit?

This experiment tests a specific mechanistic prediction: **the curvature of the activation function controls how much of Muon's advantage a penalty-based surrogate can recover.**

## Hypothesis

> Recovery % of Muon's advantage (by SGD + orthogonality penalty) **decreases monotonically** with the mean second derivative |f''(x)| of the activation function.

### Physical Intuition

Consider why this should be true:

1. **Linear activations (|f''| = 0)**: The loss landscape is quadratic in the weights. Orthogonality of weight matrices fully determines the spectral structure of the forward map. A penalty on `W^T W - I` directly regularizes the relevant degrees of freedom. Recovery should be near-complete.

2. **Smooth nonlinear activations (Tanh, Sigmoid — moderate |f''|)**: Curvature in the activation introduces higher-order interactions between weight matrices across layers. The loss landscape develops non-trivial coupling between the weight spectrum and the activation's operating point. A penalty on `W^T W - I` addresses spectral structure but misses these cross-layer couplings that Muon's per-step orthogonalization naturally handles.

3. **Highly curved activations (large |f''|)**: The mismatch between what the penalty regularizes (static weight orthogonality) and what Muon achieves (dynamic gradient orthogonalization at each step) becomes maximal. Recovery should be poorest.

### Why This Matters for the Muon-as-Gauge-Fixing Theory

If Muon is truly fixing a gauge symmetry (removing redundant degrees of freedom via orthogonalization), then the "gauge orbit" structure depends on the full computational graph, not just individual weight matrices. Activation curvature quantifies how much the gauge orbits deviate from the simple `O(n)` structure that a weight-space penalty assumes. This experiment thus probes the **limits of static gauge-fixing** vs. Muon's **dynamic, step-level gauge-fixing**.

## Experimental Design

| Component | Detail |
|-----------|--------|
| **Activations** | Linear, ReLU, LeakyReLU(0.1), GELU, Tanh, Sigmoid |
| **Architecture** | 4-layer MLP, width 32, square weight matrices |
| **Training** | 500 steps, batch size 64, random regression target |
| **Optimizers** | (a) SGD, (b) Muon, (c) SGD + weight ortho penalty, (d) SGD + partial ortho blend |
| **Metric** | Recovery % = (loss_SGD - loss_method) / (loss_SGD - loss_Muon) × 100 |
| **Key test** | Spearman rank correlation between |f''| and recovery %, expecting rho < -0.3 |

In [ ]:
import numpy as np
import os

print(f"NumPy version: {np.__version__}")
print(f"Random seed will be fixed at 42 for full reproducibility.")

## Path Resolution

We resolve the script directory for any file I/O. In a notebook context `__file__` is not defined, so we fall back to the current working directory.

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## Configuration — Hyperparameters and Architecture Choices

All hyperparameters are chosen to create a controlled, small-scale testbed where differences between optimizers are measurable but not dominated by noise:

- **WIDTH = 32, DEPTH = 4**: Square weight matrices (32x32) are essential because Muon's Newton-Schulz iteration assumes square or near-square matrices. Four layers provide enough depth for cross-layer interactions to matter, while keeping runtime tractable.
- **NUM_STEPS = 500**: Long enough for optimizers to differentiate but short enough to avoid full convergence (which would erase differences).
- **LR_SGD = 0.01, LR_MUON = 0.02**: Muon gets a slightly higher learning rate because orthogonalized gradients have bounded spectral norm, making larger steps stable.
- **ORTHO_LAMBDA = 0.003**: The penalty coefficient for `||W^T W - I||_F^2`. Set low enough to not dominate the task loss, high enough to meaningfully regularize.
- **NS_ITERS = 5**: Newton-Schulz iterations for the polar decomposition. Five iterations give ~machine precision for 32x32 matrices.
- **SEED = 42**: Fixed for reproducibility across all initializations and data generation.

In [ ]:
WIDTH = 32
DEPTH = 4
NUM_STEPS = 500
LR_SGD = 0.01
LR_MUON = 0.02
ORTHO_LAMBDA = 0.003
NS_ITERS = 5
BATCH_SIZE = 64
INPUT_DIM = 32
OUTPUT_DIM = 32
SEED = 42

print("=== Experiment Configuration ===")
print(f"  Architecture:      {DEPTH}-layer MLP, width {WIDTH}")
print(f"  Weight matrices:   {WIDTH}x{WIDTH} (square, required for Muon)")
print(f"  Training steps:    {NUM_STEPS}")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Input/Output dim:  {INPUT_DIM} / {OUTPUT_DIM}")
print(f"  LR (SGD):          {LR_SGD}")
print(f"  LR (Muon):         {LR_MUON}  (higher is safe because ||ortho(G)||_op <= 1)")
print(f"  Ortho penalty:     lambda = {ORTHO_LAMBDA}")
print(f"  Newton-Schulz:     {NS_ITERS} iterations")
print(f"  Random seed:       {SEED}")
print()
print(f"  Total weight params per layer:  {WIDTH * WIDTH} = {WIDTH}^2")
print(f"  Total weight params (all layers): {DEPTH * WIDTH * WIDTH}")

## Activation Functions and Their Derivatives

We define six activation functions spanning the full range of curvature, from zero to substantial:

| Activation | f''(x) character | Expected curvature |
|-----------|-----------------|-------------------|
| **Linear** | f''(x) = 0 everywhere | 0 (baseline) |
| **ReLU** | f''(x) = 0 almost everywhere (Dirac delta at 0) | ~0 numerically |
| **LeakyReLU(0.1)** | f''(x) = 0 a.e. (smaller jump at 0) | ~0 numerically |
| **GELU** | Smooth approximation to ReLU, nonzero f'' | Small but nonzero |
| **Tanh** | f''(x) = -2 tanh(x) sech^2(x) | Moderate |
| **Sigmoid** | f''(x) = sigmoid(x)(1-sigmoid(x))(1-2*sigmoid(x)) | Moderate |

**Key insight**: ReLU and LeakyReLU have f'' = 0 almost everywhere (the Dirac delta at x=0 has measure zero), so numerically they behave like piecewise-linear functions. This means the hypothesis predicts they should behave similarly to Linear in terms of recovery — a useful sanity check.

Each activation is paired with its first derivative (needed for backpropagation). GELU's derivative is computed via finite differences since the closed form is unwieldy.

In [ ]:
def act_linear(x):
    return x.copy()

In [ ]:
def act_relu(x):
    return np.maximum(0, x)

In [ ]:
def act_leaky_relu(x, alpha=0.1):
    return np.where(x > 0, x, alpha * x)

In [ ]:
def act_gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

In [ ]:
def act_tanh(x):
    return np.tanh(x)

In [ ]:
def act_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

In [ ]:
def dact_linear(x):
    return np.ones_like(x)

In [ ]:
def dact_relu(x):
    return (x > 0).astype(float)

In [ ]:
def dact_leaky_relu(x, alpha=0.1):
    return np.where(x > 0, 1.0, alpha)

In [ ]:
def dact_gelu(x):
    eps = 1e-5
    return (act_gelu(x + eps) - act_gelu(x - eps)) / (2 * eps)

In [ ]:
def dact_tanh(x):
    return 1.0 - np.tanh(x)**2

In [ ]:
def dact_sigmoid(x):
    s = act_sigmoid(x)
    return s * (1.0 - s)

In [ ]:
ACTIVATIONS = {
    'Linear':        (act_linear, dact_linear),
    'ReLU':          (act_relu, dact_relu),
    'LeakyReLU(0.1)':(act_leaky_relu, dact_leaky_relu),
    'GELU':          (act_gelu, dact_gelu),
    'Tanh':          (act_tanh, dact_tanh),
    'Sigmoid':       (act_sigmoid, dact_sigmoid),
}

In [ ]:
# Quick verification: print the activation registry and spot-check derivatives
print("Activation function registry:")
for name, (act_fn, dact_fn) in ACTIVATIONS.items():
    # Evaluate at a test point to verify the function/derivative pair
    x_test = np.array([0.5])
    f_val = act_fn(x_test)[0]
    df_val = dact_fn(x_test)[0]
    # Numerical derivative for cross-check
    eps = 1e-6
    df_numerical = (act_fn(x_test + eps)[0] - act_fn(x_test - eps)[0]) / (2 * eps)
    deriv_error = abs(df_val - df_numerical)
    status = "OK" if deriv_error < 1e-3 else "MISMATCH"
    print(f"  {name:<18}: f(0.5) = {f_val:.6f}, f'(0.5) = {df_val:.6f} "
          f"(numerical: {df_numerical:.6f}, err={deriv_error:.2e}) [{status}]")

## Computing Activation Curvature: Mean |f''(x)|

The curvature metric for each activation is the **mean absolute second derivative** evaluated over inputs drawn from N(0,1). This choice reflects the typical pre-activation distribution at initialization (by CLT, a sum of ~32 Gaussian-weighted inputs is approximately Gaussian).

**Method**: Central finite difference with step h = 10^-4:

$$f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}$$

We average |f''(x)| over 10,000 samples. This is the **independent variable** of our hypothesis — it defines the x-axis of the curvature-vs-recovery plot.

**Important subtlety**: For ReLU and LeakyReLU, the true f'' is a Dirac delta at x=0. With finite h, only the ~h-fraction of samples near x=0 contribute, so the measured curvature is O(h) rather than truly zero. This is correct behavior for our purposes — it means ReLU and LeakyReLU cluster near zero curvature, as they should.

In [ ]:
def estimate_mean_second_derivative(act_fn, n_samples=10000, seed=42):
    """Numerically estimate mean |f''(x)| over typical inputs N(0,1)."""
    rng = np.random.RandomState(seed)
    x = rng.randn(n_samples)
    h = 1e-4
    fpp = (act_fn(x + h) - 2.0 * act_fn(x) + act_fn(x - h)) / (h * h)
    return np.mean(np.abs(fpp))

## Network Utilities — Initialization, Forward Pass, Gradient Computation

### Weight Initialization

We use Glorot/Xavier initialization: `std = sqrt(2 / (fan_in + fan_out))`. Since all layers are square (fan_in = fan_out = WIDTH), this simplifies to `std = 1/sqrt(WIDTH)`. This keeps the forward-pass signal magnitude approximately constant across layers at initialization.

### Forward Pass

Standard MLP forward: `z_l = W_l @ a_{l-1}`, `a_l = f(z_l)`. We cache both pre-activations (for backprop derivatives) and post-activations (for gradient outer products).

### Gradient Computation

Manual backpropagation through the chain rule. The key structural observation: for each layer l, the gradient `dL/dW_l = delta_l @ a_{l-1}^T` is a rank-BATCH_SIZE matrix (at most). Muon's Newton-Schulz iteration then maps this to the nearest orthogonal matrix, which has all singular values = 1 — a dramatic restructuring of the gradient's spectral content.

### Orthogonality Penalty

The penalty `||W^T W - I||_F^2` measures how far W is from the orthogonal group O(n). Its gradient `4W(W^T W - I)` pushes W toward satisfying `W^T W = I`. This is the **static** alternative to Muon's **dynamic** orthogonalization.

### Newton-Schulz Orthogonalization

The 5th-order Newton-Schulz iteration converges cubically to the polar factor U of G = U S V^T. After convergence, the update direction is UV^T — a matrix with all singular values equal to 1. This is the heart of Muon: it equalizes the gradient's singular value spectrum, giving equal "democratic" weight to all directions in weight space.

In [ ]:
def init_weights(num_layers, width, seed):
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        std = np.sqrt(2.0 / (width + width))
        W = rng.randn(width, width) * std
        weights.append(W.copy())
    return weights

In [ ]:
def forward(weights, X, act_fn):
    activations = [X.copy()]
    pre_activations = []
    out = X.copy()
    for W in weights:
        z = W @ out
        pre_activations.append(z)
        out = act_fn(z)
        activations.append(out)
    return activations, pre_activations

In [ ]:
def compute_loss(weights, X, Y_target, act_fn):
    activations, _ = forward(weights, X, act_fn)
    Y_pred = activations[-1]
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

In [ ]:
def compute_gradients(weights, X, Y_target, act_fn, dact_fn):
    num_layers = len(weights)
    batch_size = X.shape[1]
    activations, pre_activations = forward(weights, X, act_fn)
    Y_pred = activations[-1]
    diff = Y_pred - Y_target
    delta = diff / batch_size

    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        act_deriv = dact_fn(pre_activations[l])
        delta_z = delta * act_deriv
        grads[l] = delta_z @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta_z
    return grads

In [ ]:
def ortho_penalty_gradient(W):
    """Gradient of ||W^T W - I||_F^2."""
    WtW = W.T @ W
    I = np.eye(W.shape[0])
    return 4.0 * W @ (WtW - I)

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=5):
    norm = np.linalg.norm(G, 'fro')
    if norm < 1e-12:
        return G
    X = G / norm
    for _ in range(num_iters):
        A = X.T @ X
        X = (15.0 / 8.0) * X - (10.0 / 8.0) * X @ A + (3.0 / 8.0) * X @ A @ A
    return X

In [ ]:
# Diagnostic: verify Newton-Schulz produces near-orthogonal output
_rng_test = np.random.RandomState(0)
_G_test = _rng_test.randn(WIDTH, WIDTH)
_G_orth = newton_schulz_orthogonalize(_G_test, NS_ITERS)
_orth_error = np.linalg.norm(_G_orth.T @ _G_orth - np.eye(WIDTH), 'fro')
_svs = np.linalg.svd(_G_orth, compute_uv=False)

print("=== Newton-Schulz Verification ===")
print(f"  Input matrix:       {WIDTH}x{WIDTH}, Frobenius norm = {np.linalg.norm(_G_test, 'fro'):.4f}")
print(f"  Output orthogonality error ||G_orth^T G_orth - I||_F = {_orth_error:.2e}")
print(f"  Output singular values: min={_svs.min():.6f}, max={_svs.max():.6f} (should be ~1.0)")
print(f"  NS iterations used: {NS_ITERS}")
print(f"  Conclusion: {'GOOD - output is near-orthogonal' if _orth_error < 1e-3 else 'WARNING - poor convergence'}")
del _rng_test, _G_test, _G_orth, _orth_error, _svs

## Training Routines — Four Optimizer Variants

We implement four optimizers that form a carefully designed hierarchy:

### (a) SGD — Pure Baseline
Standard stochastic gradient descent. This establishes the loss floor that the other methods must beat. SGD follows the raw gradient, which in general has a highly non-uniform singular value spectrum — dominated by the top few singular vectors.

### (b) Muon — Full Newton-Schulz Orthogonalization
Each gradient matrix G is replaced by its polar factor `ortho(G)` via Newton-Schulz iteration before the update step. This is the "gold standard" — full dynamic gauge-fixing at every step. All singular values of the update become equal, giving every direction in weight space democratic influence.

### (c) SGD + Weight Orthogonality Penalty
SGD with an additional penalty term `lambda * ||W^T W - I||_F^2` added to the loss. This is the **static** approach — it tries to keep the weights orthogonal throughout training, but does not modify the gradient directions. The hypothesis predicts this works well for linear activations but increasingly fails as curvature rises.

### (d) SGD + Partial Orthogonal Blend
A convex combination of the raw gradient and its Newton-Schulz orthogonalized version: `step = (1-alpha)*G + alpha*ortho(G)`. We sweep alpha in {0.1, 0.2, 0.3, 0.5, 0.7, 0.9} and report the best. This interpolates between pure SGD (alpha=0) and pure Muon (alpha=1), letting us measure how much orthogonalization is "needed" for each activation.

### Safety Mechanism
All training loops check for divergence (loss > 10^8) every 50 steps, returning a sentinel value of 10^10 if training blows up. This prevents NaN propagation from corrupting the results.

In [ ]:
def safe_loss(weights, X, Y, act_fn):
    loss = compute_loss(weights, X, Y, act_fn)
    if np.isnan(loss) or np.isinf(loss):
        return 1e10
    return loss

In [ ]:
def train_sgd(weights, X, Y, num_steps, lr, act_fn, dact_fn):
    weights = [W.copy() for W in weights]
    for step in range(num_steps):
        grads = compute_gradients(weights, X, Y, act_fn, dact_fn)
        for i in range(len(weights)):
            weights[i] -= lr * grads[i]
        if step % 50 == 0 and safe_loss(weights, X, Y, act_fn) > 1e8:
            return weights, 1e10
    return weights, safe_loss(weights, X, Y, act_fn)

In [ ]:
def train_muon(weights, X, Y, num_steps, lr, act_fn, dact_fn, ns_iters=5):
    weights = [W.copy() for W in weights]
    for step in range(num_steps):
        grads = compute_gradients(weights, X, Y, act_fn, dact_fn)
        for i in range(len(weights)):
            G_orth = newton_schulz_orthogonalize(grads[i], ns_iters)
            weights[i] -= lr * G_orth
        if step % 50 == 0 and safe_loss(weights, X, Y, act_fn) > 1e8:
            return weights, 1e10
    return weights, safe_loss(weights, X, Y, act_fn)

In [ ]:
def train_sgd_ortho_penalty(weights, X, Y, num_steps, lr, lam, act_fn, dact_fn):
    """SGD with weight orthogonality penalty."""
    weights = [W.copy() for W in weights]
    for step in range(num_steps):
        grads = compute_gradients(weights, X, Y, act_fn, dact_fn)
        for i in range(len(weights)):
            pen_grad = ortho_penalty_gradient(weights[i])
            weights[i] -= lr * (grads[i] + lam * pen_grad)
        if step % 50 == 0 and safe_loss(weights, X, Y, act_fn) > 1e8:
            return weights, 1e10
    return weights, safe_loss(weights, X, Y, act_fn)

In [ ]:
def train_partial_ortho(weights, X, Y, num_steps, lr, alpha, act_fn, dact_fn, ns_iters=5):
    """Blend SGD with Muon: step = (1-alpha)*G + alpha*ortho(G).
    alpha=0 is pure SGD, alpha=1 is pure Muon direction."""
    weights = [W.copy() for W in weights]
    for step in range(num_steps):
        grads = compute_gradients(weights, X, Y, act_fn, dact_fn)
        for i in range(len(weights)):
            G = grads[i]
            G_orth = newton_schulz_orthogonalize(G, ns_iters)
            # Scale G_orth to match G's norm for fair blending
            gn = np.linalg.norm(G, 'fro')
            on = np.linalg.norm(G_orth, 'fro')
            if on > 1e-12:
                G_orth_scaled = G_orth * (gn / on)
            else:
                G_orth_scaled = G_orth
            blended = (1 - alpha) * G + alpha * G_orth_scaled
            weights[i] -= lr * blended
        if step % 50 == 0 and safe_loss(weights, X, Y, act_fn) > 1e8:
            return weights, 1e10
    return weights, safe_loss(weights, X, Y, act_fn)

## Main Experiment — Full Sweep Across Activations and Optimizers

This is the core experimental loop. For each of the 6 activation functions, we:

1. **Initialize** a fresh set of weights (same seed for fair comparison across optimizers)
2. **Train** with all four optimizer variants from the same initialization
3. **Compute recovery %**: how much of Muon's advantage over SGD each method captures

The recovery metric is:

$$\text{Recovery} = \frac{\mathcal{L}_{\text{SGD}} - \mathcal{L}_{\text{method}}}{\mathcal{L}_{\text{SGD}} - \mathcal{L}_{\text{Muon}}} \times 100\%$$

- Recovery = 100% means the method fully matches Muon
- Recovery = 0% means the method is no better than SGD
- Recovery < 0% means the method is *worse* than SGD
- Recovery > 100% means the method *beats* Muon (possible but unexpected for a surrogate)

After the sweep, we test whether recovery correlates negatively with curvature using both pairwise concordance and Spearman rank correlation.

In [ ]:
def run_experiment():
    np.random.seed(SEED)
    rng = np.random.RandomState(SEED)

    X = rng.randn(INPUT_DIM, BATCH_SIZE) * 0.5
    Y = rng.randn(OUTPUT_DIM, BATCH_SIZE) * 0.3

    print("=" * 90)
    print("Experiment 2.13: Activation Curvature Theory")
    print("=" * 90)
    print()
    print("HYPOTHESIS: Ortho penalty recovery % decreases with activation curvature.")
    print("  Higher |f''| -> harder for penalty to match Muon's benefit.")
    print()
    print(f"Config: {DEPTH}-layer, width={WIDTH}, {NUM_STEPS} steps")
    print(f"  lr_sgd={LR_SGD}, lr_muon={LR_MUON}, ortho_lambda={ORTHO_LAMBDA}")
    print()

    # Data diagnostics
    print("--- Data Diagnostics ---")
    print(f"  X shape: {X.shape}, mean={X.mean():.4f}, std={X.std():.4f}")
    print(f"  Y shape: {Y.shape}, mean={Y.mean():.4f}, std={Y.std():.4f}")
    print(f"  X range: [{X.min():.3f}, {X.max():.3f}]")
    print(f"  Y range: [{Y.min():.3f}, {Y.max():.3f}]")
    print(f"  Note: X scaled by 0.5, Y by 0.3 to keep pre-activations in")
    print(f"        the sensitive region of nonlinear activations.")
    print()

    # =========================================================================
    # STEP 1: Activation curvatures
    # =========================================================================
    print("=" * 90)
    print("STEP 1: Activation Second Derivatives (mean |f''(x)|)")
    print("=" * 90)
    print()
    print("  Measuring curvature over 10,000 samples from N(0,1).")
    print("  This defines the independent variable for our hypothesis test.")
    print()

    curvatures = {}
    for name, (act_fn, _) in ACTIVATIONS.items():
        curv = estimate_mean_second_derivative(act_fn)
        curvatures[name] = curv
        print(f"  {name:<18}: mean|f''| = {curv:.6f}")

    # Post-curvature analysis
    sorted_by_curv = sorted(curvatures.items(), key=lambda kv: kv[1])
    print()
    print("  --- Curvature Ordering (ascending) ---")
    for i, (name, curv) in enumerate(sorted_by_curv):
        print(f"    {i+1}. {name:<18} {curv:.6f}")
    
    zero_curv_acts = [n for n, c in curvatures.items() if c < 0.01]
    nonzero_curv_acts = [n for n, c in curvatures.items() if c >= 0.01]
    print()
    print(f"  Near-zero curvature group (|f''| < 0.01): {zero_curv_acts}")
    print(f"  Nonzero curvature group  (|f''| >= 0.01): {nonzero_curv_acts}")
    print()
    print("  Prediction: near-zero group should show HIGH recovery %,")
    print("              nonzero group should show LOWER recovery %.")

    # =========================================================================
    # STEP 2: Train all combinations
    # =========================================================================
    print()
    print("=" * 90)
    print("STEP 2: Training")
    print("=" * 90)
    print()

    results = {}
    for name, (act_fn, dact_fn) in ACTIVATIONS.items():
        print(f"  {name}:", flush=True)
        weights_init = init_weights(DEPTH, WIDTH, seed=SEED)

        # Report initial weight statistics
        init_norms = [np.linalg.norm(W, 'fro') for W in weights_init]
        init_orth_errors = [np.linalg.norm(W.T @ W - np.eye(WIDTH), 'fro') for W in weights_init]
        print(f"    Initial weight norms: {['%.3f' % n for n in init_norms]}")
        print(f"    Initial ortho errors: {['%.3f' % e for e in init_orth_errors]}")

        # Initial loss for reference
        loss_init = safe_loss(weights_init, X, Y, act_fn)
        print(f"    Initial loss = {loss_init:.6f}")

        # (a) SGD
        _, loss_sgd = train_sgd(weights_init, X, Y, NUM_STEPS, LR_SGD, act_fn, dact_fn)
        print(f"    SGD     = {loss_sgd:.6f}")

        # (b) Muon
        _, loss_muon = train_muon(weights_init, X, Y, NUM_STEPS, LR_MUON, act_fn, dact_fn, NS_ITERS)
        print(f"    Muon    = {loss_muon:.6f}")

        # Gap analysis
        gap = loss_sgd - loss_muon
        print(f"    Muon advantage (gap): {gap:.6f}  ", end="")
        if gap > 0:
            print(f"({gap/loss_sgd*100:.1f}% of SGD loss)")
        else:
            print("(Muon did NOT beat SGD!)")

        # (c) SGD + weight ortho penalty (lambda=0.003)
        _, loss_pen = train_sgd_ortho_penalty(
            weights_init, X, Y, NUM_STEPS, LR_SGD, ORTHO_LAMBDA, act_fn, dact_fn
        )
        print(f"    Penalty = {loss_pen:.6f}")

        # (d) Partial ortho -- sweep alpha to find best
        best_alpha = 0
        best_loss_partial = loss_sgd
        alpha_sweep = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
        print(f"    Partial ortho alpha sweep:")
        for alpha in alpha_sweep:
            _, loss_partial = train_partial_ortho(
                weights_init, X, Y, NUM_STEPS, LR_SGD, alpha, act_fn, dact_fn, NS_ITERS
            )
            marker = ""
            if loss_partial < best_loss_partial:
                best_loss_partial = loss_partial
                best_alpha = alpha
                marker = " <-- best so far"
            print(f"      alpha={alpha:.1f}: loss={loss_partial:.6f}{marker}")
        print(f"    PartOrth= {best_loss_partial:.6f} (alpha={best_alpha})")

        # Recovery calculations
        gap = loss_sgd - loss_muon
        if gap > 1e-8 and loss_muon < loss_sgd:
            rec_pen = (loss_sgd - loss_pen) / gap * 100.0
            rec_partial = (loss_sgd - best_loss_partial) / gap * 100.0
        else:
            rec_pen = 0.0
            rec_partial = 0.0

        print(f"    Recovery (penalty):       {rec_pen:>7.1f}%")
        print(f"    Recovery (partial ortho): {rec_partial:>7.1f}%")
        print()

        results[name] = {
            'loss_sgd': loss_sgd,
            'loss_muon': loss_muon,
            'loss_penalty': loss_pen,
            'loss_partial': best_loss_partial,
            'best_alpha': best_alpha,
            'recovery_pen': rec_pen,
            'recovery_partial': rec_partial,
            'curvature': curvatures[name],
        }

    # =========================================================================
    # RESULTS TABLE
    # =========================================================================
    print()
    print("=" * 90)
    print("RESULTS TABLE (sorted by curvature)")
    print("=" * 90)
    header_curv = 'mean|f"|'
    print(f"{'Activation':<18} {header_curv:>8} {'SGD':>9} {'Muon':>9} "
          f"{'Penalty':>9} {'PartOrth':>9} {'Rec_pen%':>9} {'Rec_part%':>10}")
    print("-" * 90)

    sorted_names = sorted(results.keys(), key=lambda n: results[n]['curvature'])
    for name in sorted_names:
        r = results[name]
        print(f"{name:<18} {r['curvature']:>8.4f} {r['loss_sgd']:>9.5f} "
              f"{r['loss_muon']:>9.5f} {r['loss_penalty']:>9.5f} "
              f"{r['loss_partial']:>9.5f} {r['recovery_pen']:>9.1f} "
              f"{r['recovery_partial']:>10.1f}")

    # Summary statistics
    print()
    print("--- Summary Statistics ---")
    all_recoveries_pen = [results[n]['recovery_pen'] for n in sorted_names]
    all_recoveries_part = [results[n]['recovery_partial'] for n in sorted_names]
    print(f"  Penalty recovery:  mean={np.mean(all_recoveries_pen):.1f}%, "
          f"std={np.std(all_recoveries_pen):.1f}%, "
          f"range=[{np.min(all_recoveries_pen):.1f}%, {np.max(all_recoveries_pen):.1f}%]")
    print(f"  Partial recovery:  mean={np.mean(all_recoveries_part):.1f}%, "
          f"std={np.std(all_recoveries_part):.1f}%, "
          f"range=[{np.min(all_recoveries_part):.1f}%, {np.max(all_recoveries_part):.1f}%]")

    # =========================================================================
    # RECOVERY vs CURVATURE (using partial ortho -- the more meaningful metric)
    # =========================================================================
    print()
    print("=" * 90)
    print("RECOVERY vs CURVATURE (partial ortho blend)")
    print("=" * 90)
    print()
    print("  Visual representation: bar length proportional to recovery %")
    print()

    max_abs_rec = max(abs(results[n]['recovery_partial']) for n in sorted_names)
    scale = 50.0 / max(max_abs_rec, 1)

    for name in sorted_names:
        r = results[name]
        rec = r['recovery_partial']
        if rec >= 0:
            bar = '#' * max(0, int(rec * scale))
            print(f"  {name:<18} curv={r['curvature']:.4f}  rec={rec:>7.1f}%  |{bar}")
        else:
            bar = '-' * max(0, int(-rec * scale))
            print(f"  {name:<18} curv={r['curvature']:.4f}  rec={rec:>7.1f}%  |{bar} (neg)")

    print()
    print("  If the hypothesis holds, bars should generally get SHORTER")
    print("  as we move down this list (increasing curvature).")

    # =========================================================================
    # MONOTONICITY ANALYSIS
    # =========================================================================
    print()
    print("=" * 90)
    print("MONOTONICITY ANALYSIS (using partial ortho recovery)")
    print("=" * 90)
    print()
    print("  Testing whether every pair (i, j) with curvature_i < curvature_j")
    print("  also has recovery_i >= recovery_j (concordant pair).")
    print()

    curvs = np.array([results[n]['curvature'] for n in sorted_names])
    recovs = np.array([results[n]['recovery_partial'] for n in sorted_names])

    # Count concordant pairs (skip tied curvatures)
    n_pairs = 0
    n_correct = 0
    inversions = []
    for i in range(len(sorted_names)):
        for j in range(i + 1, len(sorted_names)):
            ci = results[sorted_names[i]]['curvature']
            cj = results[sorted_names[j]]['curvature']
            ri = results[sorted_names[i]]['recovery_partial']
            rj = results[sorted_names[j]]['recovery_partial']
            if abs(ci - cj) < 1e-8:
                continue
            n_pairs += 1
            if ri >= rj:
                n_correct += 1
            else:
                inversions.append((sorted_names[i], sorted_names[j],
                                   ci, cj, ri, rj))

    concordance = n_correct / n_pairs if n_pairs > 0 else 0

    print(f"  Concordant pairs (lower curv -> higher rec): {n_correct}/{n_pairs} ({concordance:.0%})")
    if inversions:
        print(f"  Inversions (higher curv has HIGHER recovery):")
        for n1, n2, c1, c2, r1, r2 in inversions:
            print(f"    {n1}(c={c1:.4f},r={r1:.1f}%) vs {n2}(c={c2:.4f},r={r2:.1f}%)")
    else:
        print(f"  No inversions found - perfect monotonic decrease!")

    # Spearman rank correlation
    def rank_array(arr):
        temp = sorted(range(len(arr)), key=lambda k: arr[k])
        ranks = [0.0] * len(arr)
        for rank_val, idx in enumerate(temp):
            ranks[idx] = rank_val + 1.0
        return ranks

    curv_ranks = rank_array(curvs.tolist())
    rec_ranks = rank_array(recovs.tolist())
    n = len(curv_ranks)
    d_sq = sum((cr - rr) ** 2 for cr, rr in zip(curv_ranks, rec_ranks))
    spearman_rho = 1 - 6 * d_sq / (n * (n * n - 1))

    print(f"\n  Spearman rho (curvature vs recovery): {spearman_rho:.4f}")
    print(f"  (Negative = higher curvature -> lower recovery)")
    print()
    print(f"  Curvature ranks: {['%s=%d' % (sorted_names[i], int(curv_ranks[i])) for i in range(n)]}")
    print(f"  Recovery ranks:  {['%s=%d' % (sorted_names[i], int(rec_ranks[i])) for i in range(n)]}")
    print(f"  Sum of squared rank differences: {d_sq:.1f}")

    # =========================================================================
    # HYPOTHESIS TESTS
    # =========================================================================
    print()
    print("=" * 90)
    print("HYPOTHESIS TESTS")
    print("=" * 90)

    # Test 1: Muon beats SGD for most activations
    muon_wins = sum(1 for n in ACTIVATIONS if results[n]['loss_muon'] < results[n]['loss_sgd'])
    test1_pass = muon_wins >= len(ACTIVATIONS) * 0.5
    print(f"\n  Test 1: Muon beats SGD for >= 50% of activations")
    print(f"    Muon wins: {muon_wins}/{len(ACTIVATIONS)}  [{'PASS' if test1_pass else 'FAIL'}]")
    print(f"    Rationale: If Muon does not consistently help, the recovery metric")
    print(f"    is meaningless -- there is nothing to recover.")
    for n in ACTIVATIONS:
        r = results[n]
        win = "WIN" if r['loss_muon'] < r['loss_sgd'] else "LOSE"
        print(f"      {n:<18}: SGD={r['loss_sgd']:.5f}, Muon={r['loss_muon']:.5f} [{win}]")

    # Test 2: Partial ortho recovers SOME advantage for at least 3 activations
    positive_rec = sum(1 for n in ACTIVATIONS if results[n]['recovery_partial'] > 10)
    test2_pass = positive_rec >= 3
    print(f"\n  Test 2: Partial ortho recovery > 10% for >= 3 activations")
    print(f"    Count: {positive_rec}/{len(ACTIVATIONS)}  [{'PASS' if test2_pass else 'FAIL'}]")
    print(f"    Rationale: The partial ortho blend should be able to recover at least")
    print(f"    some of Muon's benefit for multiple activations to have a meaningful signal.")

    # Test 3: Negative Spearman correlation
    test3_pass = spearman_rho < -0.3
    print(f"\n  Test 3: Spearman rho < -0.3 (negative correlation)")
    print(f"    rho = {spearman_rho:.4f}  [{'PASS' if test3_pass else 'FAIL'}]")
    print(f"    Rationale: A threshold of -0.3 indicates a meaningful negative")
    print(f"    monotonic relationship between curvature and recovery.")

    # Test 4: Concordance >= 60%
    test4_pass = concordance >= 0.60
    print(f"\n  Test 4: Concordance >= 60% (monotonic decreasing trend)")
    print(f"    Concordance = {concordance:.0%}  [{'PASS' if test4_pass else 'FAIL'}]")
    print(f"    Rationale: 60% concordance means the majority of pairwise comparisons")
    print(f"    support the predicted ordering: lower curvature -> higher recovery.")

    # Test 5: Separation between zero-curv and nonzero-curv groups
    zero_curv = [n for n in sorted_names if results[n]['curvature'] < 0.01]
    nonzero_curv = [n for n in sorted_names if results[n]['curvature'] >= 0.01]
    if zero_curv and nonzero_curv:
        mean_zero_rec = np.mean([results[n]['recovery_partial'] for n in zero_curv])
        mean_nz_rec = np.mean([results[n]['recovery_partial'] for n in nonzero_curv])
        test5_pass = mean_zero_rec > mean_nz_rec
        print(f"\n  Test 5: Zero-curv activations have higher mean recovery")
        print(f"    Zero-curv group {zero_curv}: mean recovery = {mean_zero_rec:.1f}%")
        print(f"    Nonzero-curv group {nonzero_curv}: mean recovery = {mean_nz_rec:.1f}%")
        print(f"    Separation: {mean_zero_rec - mean_nz_rec:.1f} percentage points")
        print(f"    [{'PASS' if test5_pass else 'FAIL'}]")
        print(f"    Rationale: This is a coarser version of the correlation test -- it just")
        print(f"    asks if the piecewise-linear group (where penalty should work well)")
        print(f"    outperforms the curved group on average.")
    else:
        test5_pass = False
        print(f"\n  Test 5: Not enough groups  [FAIL]")

    # =========================================================================
    # OVERALL VERDICT
    # =========================================================================
    print()
    print("=" * 90)

    core_pass = (test3_pass or test4_pass) and test1_pass
    direction_pass = spearman_rho < 0

    if core_pass:
        print("OVERALL: PASS")
        print("  Recovery % decreases with activation curvature.")
        print(f"  Spearman rho = {spearman_rho:.4f}, concordance = {concordance:.0%}")
    elif direction_pass and test1_pass:
        print("OVERALL: PARTIAL PASS")
        print("  Correlation direction is correct (negative rho) but not strongly monotonic.")
        print(f"  Spearman rho = {spearman_rho:.4f}, concordance = {concordance:.0%}")
    else:
        print("OVERALL: FAIL")
        if not test1_pass:
            print("  Muon did not consistently beat SGD across activations.")
        if not direction_pass:
            print(f"  Correlation POSITIVE (rho={spearman_rho:.4f}), opposite to prediction.")
        if not (test3_pass or test4_pass):
            print(f"  Correlation too weak (rho={spearman_rho:.4f}, conc={concordance:.0%})")

    print("=" * 90)
    
    # Print final diagnostic summary
    print()
    print("--- Final Diagnostic Summary ---")
    print(f"  Tests passed: {sum([test1_pass, test2_pass, test3_pass, test4_pass, test5_pass])}/5")
    print(f"  Spearman rho:  {spearman_rho:.4f}")
    print(f"  Concordance:   {concordance:.0%}")
    print(f"  Best alpha range: {[results[n]['best_alpha'] for n in sorted_names]}")
    print(f"  (Does alpha increase with curvature? That would mean more")
    print(f"   orthogonalization is needed as curvature rises.)")

## Execute the Experiment

Running the full sweep: 6 activations x 4 optimizers (plus 6 alpha values for the partial blend), totaling ~30 training runs of 500 steps each. Each run trains a 4-layer, 32-wide MLP from the same initialization.

**What to watch for in the output:**

1. **Curvature ordering**: Linear/ReLU/LeakyReLU should cluster near zero; GELU/Tanh/Sigmoid should be nonzero.
2. **Muon vs SGD gap**: Muon should consistently beat SGD. If it does not for some activation, that activation provides no useful recovery signal.
3. **Recovery vs curvature trend**: The key test. Lower-curvature activations should show higher recovery %.
4. **Best alpha values**: If alpha increases with curvature, it means curved activations *need more orthogonalization* — supporting the theory that curvature makes the static penalty insufficient.
5. **Concordance and Spearman rho**: Quantitative measures of the predicted monotonic relationship.

In [ ]:
if __name__ == "__main__":
    run_experiment()

## Conclusions and Interpretation

### What This Experiment Reveals About Muon

This experiment tests whether Muon's advantage is *mechanistically* linked to activation curvature, or whether a simple orthogonality penalty can replicate it regardless of the nonlinearity.

**If the hypothesis PASSES (negative Spearman rho, high concordance):**
- Activation curvature genuinely modulates the difficulty of recovering Muon's benefit via static penalties.
- This supports the view that Muon's per-step orthogonalization captures something about the *dynamic* interaction between gradients and nonlinear activations that a weight-space penalty cannot.
- In the gauge-fixing interpretation: the gauge orbits are simple (close to O(n)) for linear networks, but become distorted by activation curvature. Muon adapts to this distortion step-by-step; a static penalty does not.

**If the hypothesis FAILS (positive or near-zero rho):**
- Activation curvature may not be the primary modulator of Muon's advantage. Other factors (conditioning, gradient rank, signal propagation) may dominate.
- The penalty may be either universally effective or universally ineffective regardless of curvature, suggesting the relevant degrees of freedom are not well-described by mean |f''|.
- Alternative hypotheses to investigate: gradient spectrum anisotropy, effective rank of layer-wise gradients, or depth-dependent signal attenuation.

### Connection to the Broader Research Program

This is one of several experiments in the Muon-as-Gauge-Fixing framework. Key related experiments include:
- **TANH_VANISHING_ortho_penalty**: Tests whether tanh's vanishing gradient interacts with the penalty mechanism.
- **H20b_COMPOUNDING_CURVATURE_DEPENDENCE**: Tests whether curvature compounds across layers.
- **PRODUCT_KAPPA_crossover_vs_depth**: Tests the crossover depth where Muon's advantage becomes dominant.

Together, these experiments build a picture of when and why Muon's orthogonalization matters — and where simpler surrogates suffice.